In [ ]:
# Cell 1: Setup and data loading
import json, os, re, csv, time, torch
from collections import Counter, defaultdict
from pathlib import Path

DATA_DIR = Path("/content/drive/MyDrive/candidate_pack/data")
SAVE_DIR = Path("/content/drive/MyDrive/candidate_pack/outputs")
SAVE_DIR.mkdir(parents=True, exist_ok=True)

def load_jsonl(path):
    with open(path, encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

# Load original splits
train_raw = load_jsonl(DATA_DIR / "train.jsonl")
dev        = load_jsonl(DATA_DIR / "dev.jsonl")
test       = load_jsonl(DATA_DIR / "test_inputs.jsonl")

# Load the already-cleaned training data we saved earlier
cleaned_train = load_jsonl(SAVE_DIR / "train_cleaned.jsonl")

print(f"train_raw: {len(train_raw)}  cleaned_train: {len(cleaned_train)}")
print(f"dev: {len(dev)}  test: {len(test)}")

train_raw: 1445  cleaned_train: 1411
dev: 200  test: 400


In [ ]:
# Cell 2: Parse all labels, check JSON validity and key sets

KEYS = ["category", "priority", "amount_paise", "txn_date", "txn_id", "channel", "language", "needs_human"]
CATEGORIES = {"payment_failed", "refund", "fraud", "account_access", "kyc", "offers", "other"}
PRIORITIES = {"P1", "P2", "P3"}
CHANNELS = {"upi", "card", "netbanking", "wallet", None}
LANGUAGES = {"en", "hi-en"}

issues = []  # collect (id, issue_type, detail)

for row in train_raw:
    rid = row["id"]
    asst = row["messages"][2]["content"]

    # 1. JSON validity
    try:
        label = json.loads(asst)
    except json.JSONDecodeError as e:
        issues.append((rid, "invalid_json", str(e)))
        continue

    # 2. Key set
    if set(label.keys()) != set(KEYS):
        extra = set(label.keys()) - set(KEYS)
        missing = set(KEYS) - set(label.keys())
        issues.append((rid, "wrong_keys", f"extra={extra}, missing={missing}"))
        continue

    # 3. Type checks
    if label["category"] not in CATEGORIES:
        issues.append((rid, "bad_category", label["category"]))
    if label["priority"] not in PRIORITIES:
        issues.append((rid, "bad_priority", label["priority"]))
    if label["channel"] not in CHANNELS:
        issues.append((rid, "bad_channel", label["channel"]))
    if label["language"] not in LANGUAGES:
        issues.append((rid, "bad_language", label["language"]))
    if not isinstance(label["needs_human"], bool):
        issues.append((rid, "bad_needs_human", f"type={type(label['needs_human']).__name__}, val={label['needs_human']}"))
    if label["amount_paise"] is not None and not isinstance(label["amount_paise"], int):
        issues.append((rid, "bad_amount_type", f"type={type(label['amount_paise']).__name__}, val={label['amount_paise']}"))
    if label["txn_id"] is not None:
        if not re.fullmatch(r"NP\d{10}", str(label["txn_id"])):
            issues.append((rid, "bad_txn_id_format", label["txn_id"]))
    if label["txn_date"] is not None:
        if not re.fullmatch(r"\d{4}-\d{2}-\d{2}", str(label["txn_date"])):
            issues.append((rid, "bad_txn_date_format", label["txn_date"]))

    # 4. Null rules: account_access, kyc, other must have null txn fields
    if label["category"] in ("account_access", "kyc", "other"):
        for fld in ("amount_paise", "txn_date", "txn_id", "channel"):
            if label[fld] is not None:
                issues.append((rid, "txn_field_should_be_null", f"{fld}={label[fld]} but category={label['category']}"))

print(f"Total issues found: {len(issues)}")
print(f"Rows with issues: {len(set(i[0] for i in issues))}")
print()
issue_counts = Counter(i[1] for i in issues)
for itype, cnt in issue_counts.most_common():
    print(f"  {itype}: {cnt}")
print()
print("First 20 issues:")
for rid, itype, detail in issues[:20]:
    print(f"  {rid:10s}  {itype:28s}  {detail[:80]}")

Total issues found: 115
Rows with issues: 115

  wrong_keys: 70
  invalid_json: 25
  bad_txn_id_format: 20

First 20 issues:
  tr-00020    wrong_keys                    extra={'prio'}, missing={'priority'}
  tr-00049    wrong_keys                    extra={'prio'}, missing={'priority'}
  tr-00058    wrong_keys                    extra={'prio'}, missing={'priority'}
  tr-00060    invalid_json                  Unterminated string starting at: line 1 column 79 (char 78)
  tr-00071    bad_txn_id_format             NP-9718716109
  tr-00072    wrong_keys                    extra={'prio'}, missing={'priority'}
  tr-00077    wrong_keys                    extra={'prio'}, missing={'priority'}
  tr-00095    invalid_json                  Unterminated string starting at: line 1 column 97 (char 96)
  tr-00114    wrong_keys                    extra={'prio'}, missing={'priority'}
  tr-00120    wrong_keys                    extra={'prio'}, missing={'priority'}
  tr-00123    wrong_keys                  

In [ ]:
# Cell 3: Inspect the three issue types

# --- 1. wrong_keys: "prio" instead of "priority" ---
prio_rows = [r for r in train if r["id"] in {i[0] for i in issues if i[1] == "wrong_keys"}]
print("=== WRONG KEYS: sample ===")
sample = json.loads(prio_rows[0]["messages"][2]["content"])
print(json.dumps(sample, indent=2))
print(f"\nAll have 'prio' instead of 'priority'? ", end="")
all_prio = all("prio" in json.loads(r["messages"][2]["content"]) for r in prio_rows)
print(all_prio)
# Check if prio values are valid priorities
prio_vals = Counter(json.loads(r["messages"][2]["content"])["prio"] for r in prio_rows)
print(f"prio values: {prio_vals}")

# --- 2. invalid_json: truncated strings ---
print("\n=== INVALID JSON: samples ===")
bad_json_ids = {i[0] for i in issues if i[1] == "invalid_json"}
for row in train:
    if row["id"] in bad_json_ids:
        print(f"\n{row['id']}:")
        print(f"  label: {row['messages'][2]['content']}")
        if len(bad_json_ids) > 5:
            # just show first 5
            bad_json_ids.discard(row["id"])
            if len(bad_json_ids) <= 20:
                continue
            break

# --- 3. bad_txn_id_format ---
print("\n=== BAD TXN_ID FORMAT: all values ===")
bad_txn_ids = [(i[0], i[2]) for i in issues if i[1] == "bad_txn_id_format"]
for rid, val in bad_txn_ids[:15]:
    print(f"  {rid}: {val}")

=== WRONG KEYS: sample ===
{
  "category": "refund",
  "prio": "low",
  "amount_paise": 97800,
  "txn_date": "02/09/2026",
  "txn_id": null,
  "channel": "wallet",
  "language": "en",
  "needs_human": true
}

All have 'prio' instead of 'priority'? True
prio values: Counter({'low': 44, 'medium': 15, 'high': 11})

=== INVALID JSON: samples ===

tr-00060:
  label: {"category": "kyc", "priority": "P3", "amount_paise": null, "txn_date": null, "t

=== BAD TXN_ID FORMAT: all values ===
  tr-00071: NP-9718716109
  tr-00147: np9891267518
  tr-00151: NP-2339259875
  tr-00391: NP 9255627510
  tr-00420: np 81633 47892
  tr-00460: NP 6949410784
  tr-00470: np 11245 46709
  tr-00570: np 87267 97611
  tr-00595: np3247474643
  tr-00649: NP-8919677483
  tr-00771: np9038808761
  tr-00838: NP 6186175115
  tr-00888: np7786684387
  tr-00899: NP-1325500121
  tr-00908: np 45856 96868


In [ ]:
# Cell 4: Understand the v1 (prio) rows fully

prio_rows = [r for r in train if r["id"] in
    {i[0] for i in issues if i[1] == "wrong_keys"}]

# Check date format in v1 rows
v1_dates = []
for r in prio_rows:
    label = json.loads(r["messages"][2]["content"])
    v1_dates.append(label.get("txn_date"))

date_formats = Counter()
for d in v1_dates:
    if d is None:
        date_formats["null"] += 1
    elif re.fullmatch(r"\d{2}/\d{2}/\d{4}", d):
        date_formats["DD/MM/YYYY"] += 1
    elif re.fullmatch(r"\d{4}-\d{2}-\d{2}", d):
        date_formats["YYYY-MM-DD"] += 1
    else:
        date_formats[f"other: {d}"] += 1

print("Date formats in v1 rows:", date_formats)

# Check what "low/medium/high" should map to
# We can't just map blindly - we need to recompute priority from rules
# But first, let's see the category distribution
v1_cats = Counter(json.loads(r["messages"][2]["content"])["category"] for r in prio_rows)
print(f"\nv1 categories: {v1_cats}")

# Check needs_human in v1 rows
v1_nh = Counter(json.loads(r["messages"][2]["content"])["needs_human"] for r in prio_rows)
print(f"v1 needs_human: {v1_nh}")

# Check if high=P1, medium=P2, low=P3 makes sense
print("\nv1 prio vs category:")
for r in prio_rows[:10]:
    label = json.loads(r["messages"][2]["content"])
    cat = label["category"]
    prio = label["prio"]
    amt = label["amount_paise"]
    print(f"  {r['id']}: cat={cat:18s} prio={prio:7s} amt={amt}")

Date formats in v1 rows: Counter({'null': 40, 'DD/MM/YYYY': 30})

v1 categories: Counter({'refund': 18, 'payment_failed': 17, 'other': 9, 'account_access': 9, 'kyc': 7, 'offers': 7, 'fraud': 3})
v1 needs_human: Counter({False: 50, True: 20})

v1 prio vs category:
  tr-00020: cat=refund             prio=low     amt=97800
  tr-00049: cat=other              prio=low     amt=None
  tr-00058: cat=payment_failed     prio=low     amt=None
  tr-00072: cat=kyc                prio=low     amt=None
  tr-00077: cat=refund             prio=medium  amt=3700699
  tr-00114: cat=other              prio=low     amt=None
  tr-00120: cat=other              prio=low     amt=None
  tr-00123: cat=refund             prio=high    amt=47700
  tr-00153: cat=refund             prio=low     amt=229000
  tr-00160: cat=refund             prio=high    amt=22810899


In [ ]:
# Cell 5: Full cleaning pipeline

import csv
from datetime import datetime, timedelta

cleaning_log = []  # (id, action, reason)

def parse_ticket(row):
    """Extract metadata from the user message."""
    text = row["messages"][1]["content"]
    # Received date
    m = re.search(r"Received:\s*(\d{4}-\d{2}-\d{2})", text)
    received = m.group(1) if m else None
    # Source
    m = re.search(r"Source:\s*(\S+)", text)
    source = m.group(1) if m else None
    # Body (after the two header lines)
    lines = text.strip().split("\n")
    body_start = 0
    for i, line in enumerate(lines):
        if line.strip() == "":
            body_start = i + 1
            break
    body = "\n".join(lines[body_start:]).strip()
    return {"received": received, "source": source, "body": body, "full_text": text}

def has_legal_threat(text):
    """Check for legal threat keywords."""
    t = text.lower()
    patterns = [
        r'\brbi\b', r'\bombudsman\b', r'\bconsumer\s*court\b',
        r'\blawyer\b', r'\blegal\s*notice\b', r'\blegal\s*action\b',
        r'\bcourt\b', r'\badvocate\b'
    ]
    return any(re.search(p, t) for p in patterns)

def has_repeat_contact(text):
    """Check for 'contacted before' signals."""
    t = text.lower()
    patterns = [
        r'\bthird\s*time\b', r'\bagain\s*and\s*again\b',
        r'\bbaar\s*baar\b', r'\bmultiple\s*times?\b',
        r'\bso\s*many\s*times?\b', r'\bfollowing\s*up\b',
        r'\bfollow\s*up\b', r'\bsecond\s*time\b',
        r'\bcontacted\s*(you|support|already|before|earlier)\b',
        r'\bpreviously\s*(raised|contacted|reported|called|complained)\b',
        r'\balready\s*(raised|contacted|reported|called|complained|written|mailed|emailed)\b',
        r'\bfourth\s*time\b', r'\bfifth\s*time\b',
        r'\bno\s*(response|reply|resolution)\b',
        r'\bpehle\s*bhi\b', r'\bdobara\b', r'\bphir\s*se\b',
    ]
    return any(re.search(p, t) for p in patterns)

def compute_priority(category, amount_paise, text):
    """Recompute priority from schema rules."""
    # P1 conditions
    if category == "fraud":
        return "P1"
    if amount_paise is not None and amount_paise >= 5000000:  # 50,000 rupees
        return "P1"
    if has_legal_threat(text):
        return "P1"
    # P2 conditions
    if category in ("payment_failed", "refund") and amount_paise is not None and amount_paise >= 500000:
        return "P2"
    if category == "account_access":
        return "P2"
    # P3
    return "P3"

def compute_needs_human(priority, text):
    """Recompute needs_human from schema rules."""
    if priority == "P1":
        return True
    if has_repeat_contact(text):
        return True
    return False

def fix_date_ddmmyyyy(date_str):
    """Convert DD/MM/YYYY to YYYY-MM-DD."""
    if date_str is None:
        return None
    m = re.fullmatch(r"(\d{2})/(\d{2})/(\d{4})", date_str)
    if m:
        dd, mm, yyyy = m.groups()
        return f"{yyyy}-{mm}-{dd}"
    return date_str

def normalize_txn_id(tid):
    """NP + 10 digits, uppercase, no spaces/hyphens."""
    if tid is None:
        return None
    cleaned = re.sub(r"[\s\-]", "", tid).upper()
    if re.fullmatch(r"NP\d{10}", cleaned):
        return cleaned
    return tid  # return as-is if can't normalize

# ============================================================
# PASS 1: Fix what we can, drop what we can't
# ============================================================

cleaned_train = []
dropped_ids = set()

for row in train:
    rid = row["id"]
    asst = row["messages"][2]["content"]
    ticket = parse_ticket(row)

    # Try to parse JSON
    try:
        label = json.loads(asst)
    except json.JSONDecodeError:
        cleaning_log.append((rid, "drop", "invalid/truncated JSON"))
        dropped_ids.add(rid)
        continue

    if not isinstance(label, dict):
        cleaning_log.append((rid, "drop", "label is not a dict"))
        dropped_ids.add(rid)
        continue

    fixes_applied = []

    # --- Fix v1 rows (prio key) ---
    if "prio" in label and "priority" not in label:
        old_prio = label.pop("prio")
        # Recompute priority from rules
        new_priority = compute_priority(
            label.get("category"),
            label.get("amount_paise"),
            ticket["full_text"]
        )
        label["priority"] = new_priority
        fixes_applied.append(f"prio={old_prio} -> priority={new_priority}")

        # Fix DD/MM/YYYY dates
        if label.get("txn_date") and "/" in str(label.get("txn_date", "")):
            old_date = label["txn_date"]
            label["txn_date"] = fix_date_ddmmyyyy(old_date)
            fixes_applied.append(f"date {old_date} -> {label['txn_date']}")

        # Recompute needs_human since priority changed
        old_nh = label["needs_human"]
        label["needs_human"] = compute_needs_human(label["priority"], ticket["full_text"])
        if old_nh != label["needs_human"]:
            fixes_applied.append(f"needs_human {old_nh} -> {label['needs_human']}")

    # --- Fix bad txn_id format ---
    if label.get("txn_id") is not None:
        old_tid = label["txn_id"]
        new_tid = normalize_txn_id(old_tid)
        if new_tid != old_tid:
            if re.fullmatch(r"NP\d{10}", new_tid):
                label["txn_id"] = new_tid
                fixes_applied.append(f"txn_id {old_tid} -> {new_tid}")
            else:
                # Can't normalize, leave it (will catch in validation)
                pass

    # --- Rewrite the assistant message ---
    if fixes_applied:
        row = {**row}  # shallow copy
        row["messages"] = list(row["messages"])
        row["messages"][2] = {
            "role": "assistant",
            "content": json.dumps(label, ensure_ascii=False)
        }
        cleaning_log.append((rid, "fix", "; ".join(fixes_applied)))

    cleaned_train.append(row)

print(f"Dropped: {len(dropped_ids)}")
print(f"Fixed:   {len([l for l in cleaning_log if l[1] == 'fix'])}")
print(f"Remaining: {len(cleaned_train)}")
print(f"\nFix breakdown:")
for rid, action, reason in cleaning_log[:15]:
    print(f"  {rid:10s} {action:5s} {reason[:80]}")

Dropped: 25
Fixed:   90
Remaining: 1420

Fix breakdown:
  tr-00020   fix   prio=low -> priority=P3; date 02/09/2026 -> 2026-09-02
  tr-00049   fix   prio=low -> priority=P3
  tr-00058   fix   prio=low -> priority=P3; date 26/08/2026 -> 2026-08-26
  tr-00060   drop  invalid/truncated JSON
  tr-00071   fix   txn_id NP-9718716109 -> NP9718716109
  tr-00072   fix   prio=low -> priority=P3; needs_human True -> False
  tr-00077   fix   prio=medium -> priority=P2; date 29/07/2026 -> 2026-07-29
  tr-00095   drop  invalid/truncated JSON
  tr-00114   fix   prio=low -> priority=P3
  tr-00120   fix   prio=low -> priority=P3
  tr-00123   fix   prio=high -> priority=P1; date 21/07/2026 -> 2026-07-21
  tr-00125   drop  invalid/truncated JSON
  tr-00147   fix   txn_id np9891267518 -> NP9891267518
  tr-00151   fix   txn_id NP-2339259875 -> NP2339259875
  tr-00153   fix   prio=low -> priority=P3; date 04/09/2026 -> 2026-09-04


In [ ]:
# Cell 6: Validate cleaned data against schema rules

pass2_issues = []

for row in cleaned_train:
    rid = row["id"]
    label = json.loads(row["messages"][2]["content"])
    ticket = parse_ticket(row)

    # --- Priority check ---
    expected_pri = compute_priority(
        label["category"], label["amount_paise"], ticket["full_text"]
    )
    if label["priority"] != expected_pri:
        pass2_issues.append((rid, "wrong_priority",
            f"got={label['priority']} expected={expected_pri} cat={label['category']} amt={label['amount_paise']}"))

    # --- needs_human check ---
    expected_nh = compute_needs_human(label["priority"], ticket["full_text"])
    if label["needs_human"] != expected_nh:
        pass2_issues.append((rid, "wrong_needs_human",
            f"got={label['needs_human']} expected={expected_nh} pri={label['priority']}"))

    # --- Null rules for non-txn categories ---
    if label["category"] in ("account_access", "kyc", "other"):
        for fld in ("amount_paise", "txn_date", "txn_id", "channel"):
            if label[fld] is not None:
                pass2_issues.append((rid, "should_be_null",
                    f"{fld}={label[fld]} but cat={label['category']}"))

    # --- txn_id format (after normalization) ---
    if label["txn_id"] is not None:
        if not re.fullmatch(r"NP\d{10}", str(label["txn_id"])):
            pass2_issues.append((rid, "bad_txn_id", str(label["txn_id"])))

    # --- txn_date format ---
    if label["txn_date"] is not None:
        if not re.fullmatch(r"\d{4}-\d{2}-\d{2}", str(label["txn_date"])):
            pass2_issues.append((rid, "bad_date_format", str(label["txn_date"])))

    # --- amount_paise type ---
    if label["amount_paise"] is not None:
        if not isinstance(label["amount_paise"], int):
            pass2_issues.append((rid, "amount_not_int",
                f"type={type(label['amount_paise']).__name__}"))

    # --- channel ---
    if label["channel"] not in {"upi", "card", "netbanking", "wallet", None}:
        pass2_issues.append((rid, "bad_channel", str(label["channel"])))

    # --- category ---
    if label["category"] not in CATEGORIES:
        pass2_issues.append((rid, "bad_category", str(label["category"])))

    # --- language: quick Hinglish heuristic ---
    body = ticket["body"].lower()
    hi_words = {"hai", "hain", "nahi", "kya", "mera", "meri", "mere",
                "kaise", "karo", "karna", "karein", "karke", "hua",
                "nhi", "bhi", "aur", "abhi", "bol", "bolo",
                "paisa", "paise", "rupaye", "wala", "wali", "raha",
                "rahi", "tha", "thi", "ho", "hota", "baar",
                "kab", "kyun", "kyu", "krna", "krke", "hoga",
                "huaa", "gaya", "gayi", "aaya", "aayi", "mujhe",
                "maine", "tumhe", "apna", "apni", "yeh", "woh",
                "koi", "kuch", "sab", "bahut", "bohot"}
    words = set(re.findall(r"[a-z]+", body))
    hi_count = len(words & hi_words)
    detected_lang = "hi-en" if hi_count >= 3 else "en"
    if label["language"] != detected_lang:
        pass2_issues.append((rid, "possible_lang_mismatch",
            f"label={label['language']} detected={detected_lang} hi_words={hi_count}"))

print(f"Pass 2 issues: {len(pass2_issues)}")
print(f"Rows affected: {len(set(i[0] for i in pass2_issues))}")
print()
issue_counts2 = Counter(i[1] for i in pass2_issues)
for itype, cnt in issue_counts2.most_common():
    print(f"  {itype}: {cnt}")
print()
print("All issues:")
for rid, itype, detail in pass2_issues:
    print(f"  {rid:10s} {itype:24s} {detail[:90]}")

Pass 2 issues: 176
Rows affected: 131

  wrong_needs_human: 82
  wrong_priority: 65
  possible_lang_mismatch: 29

All issues:
  tr-00013   wrong_priority           got=P1 expected=P3 cat=kyc amt=None
  tr-00018   wrong_priority           got=P1 expected=P3 cat=offers amt=25000
  tr-00018   wrong_needs_human        got=False expected=True pri=P1
  tr-00023   wrong_priority           got=P1 expected=P3 cat=kyc amt=None
  tr-00023   wrong_needs_human        got=False expected=True pri=P1
  tr-00028   possible_lang_mismatch   label=hi-en detected=en hi_words=2
  tr-00031   wrong_needs_human        got=True expected=False pri=P2
  tr-00034   possible_lang_mismatch   label=hi-en detected=en hi_words=0
  tr-00045   wrong_priority           got=P3 expected=P1 cat=fraud amt=1159400
  tr-00045   wrong_needs_human        got=True expected=False pri=P3
  tr-00047   wrong_needs_human        got=True expected=False pri=P3
  tr-00049   possible_lang_mismatch   label=hi-en detected=en hi_words=1
  tr-

In [ ]:
# Cell 7: Spot-check priority, needs_human, and language issues

# --- 1. Wrong priority: check if OUR detector is missing something ---
print("=== WRONG PRIORITY SAMPLES (checking ticket text) ===\n")
pri_issues = [(rid, detail) for rid, itype, detail in pass2_issues if itype == "wrong_priority"]
for rid, detail in pri_issues[:8]:
    row = next(r for r in cleaned_train if r["id"] == rid)
    label = json.loads(row["messages"][2]["content"])
    ticket = parse_ticket(row)
    # Show just enough to see if there's a legal threat or high amount
    print(f"{rid}: {detail}")
    print(f"  Ticket (last 300 chars): ...{ticket['body'][-300:]}")
    print()

# --- 2. needs_human: cases where we expect True but got False ---
print("\n=== NEEDS_HUMAN: got=True expected=False (is there repeat contact?) ===\n")
nh_false_positive = [(rid, detail) for rid, itype, detail in pass2_issues
                     if itype == "wrong_needs_human" and "got=True expected=False" in detail]
for rid, detail in nh_false_positive[:5]:
    row = next(r for r in cleaned_train if r["id"] == rid)
    ticket = parse_ticket(row)
    print(f"{rid}: {detail}")
    print(f"  Ticket (last 200 chars): ...{ticket['body'][-200:]}")
    print()

# --- 3. Language: check the "mismatch" cases ---
print("\n=== LANGUAGE MISMATCH SAMPLES ===\n")
lang_issues = [(rid, detail) for rid, itype, detail in pass2_issues if itype == "possible_lang_mismatch"]
for rid, detail in lang_issues[:8]:
    row = next(r for r in cleaned_train if r["id"] == rid)
    ticket = parse_ticket(row)
    print(f"{rid}: {detail}")
    print(f"  Body: {ticket['body'][:250]}")
    print()

=== WRONG PRIORITY SAMPLES (checking ticket text) ===

tr-00013: got=P1 expected=P3 cat=kyc amt=None
  Ticket (last 300 chars): ...Subject: Issue with my account

Hi,
KYC mein address update karna hai. Android app use kar raha hoon. Bank ref 782509120860 hai. Pehle bhi complaint kar chuka hoon.

Regards,
Meera Joshi

tr-00018: got=P1 expected=P3 cat=offers amt=25000
  Ticket (last 300 chars): ...Subject: Issue with my account

Dear NimbusPay support,
Referral bonus was not received after my friend signed up. Mode: UPI. My phone is a OnePlus Nord. The transaction amount is Rs. 250. Thanks.

Regards,
Imran Khan

tr-00023: got=P1 expected=P3 cat=kyc amt=None
  Ticket (last 300 chars): ...URGENT!! How can you assist on verifying my identity? I am on the iOS app. Please look into it.

tr-00045: got=P3 expected=P1 cat=fraud amt=1159400
  Ticket (last 300 chars): ...Subject: Help needed


I AM MISSING CASH FROM MY BANK ACCOUNT ACCORDING TO MY APP. The amount was INR 11,594. I am on the iOS ap

In [ ]:
# Cell 8: Improved detectors + full re-clean

# Better repeat-contact detector
def has_repeat_contact(text):
    t = text.lower()
    patterns = [
        r'\bthird\s*time\b', r'\bsecond\s*time\b', r'\bfourth\s*time\b', r'\bfifth\s*time\b',
        r'\bagain\s*and\s*again\b',
        r'\bbaar\s*baar\b', r'\bteesri\s*baar\b', r'\bdoosri\s*baar\b',
        r'\bmultiple\s*times?\b', r'\bso\s*many\s*times?\b',
        r'\bfollow\s*up\b', r'\bfollowing\s*up\b',
        r'\bsecond\s*complaint\b', r'\bthird\s*complaint\b',
        r'\b(contacted|called|written|emailed|complained|raised|reported)\s*(before|earlier|already|previously)\b',
        r'\b(already|previously)\s*(contacted|called|written|emailed|complained|raised|reported)\b',
        r'\bno\s*(response|reply|resolution)\s*(yet|still|till now|from)?\b',
        r'\bpehle\s*bhi\b', r'\bdobara\b', r'\bphir\s*se\b',
        r'\bsame\s*issue\b', r'\bsame\s*problem\b',
    ]
    return any(re.search(p, t) for p in patterns)

# Better Hinglish detector
def detect_language(text):
    t = text.lower()
    hi_words = {
        "hai", "hain", "nahi", "nhi", "kya", "kaise", "karo", "karna",
        "karein", "karke", "hua", "bhi", "aur", "abhi", "paisa", "paise",
        "rupaye", "wala", "wali", "raha", "rahi", "tha", "thi", "ho",
        "hota", "baar", "kab", "kyun", "kyu", "krna", "krke", "hoga",
        "huaa", "gaya", "gayi", "aaya", "aayi", "mujhe", "maine",
        "apna", "apni", "yeh", "woh", "koi", "kuch", "sab", "bahut",
        "bohot", "mera", "meri", "mere", "tumhe", "bol", "bolo",
        "kar", "karun", "karo", "lagega", "lagta", "dekho", "dekh",
        "hua", "hui", "hue", "ruka", "ruki", "atka",
        "nahi", "nahin", "mat", "band", "chalu",
        "sir", "namaste", "dhanyavaad", "shukriya",
        "lekin", "isliye", "kyunki", "toh", "par",
        "kal", "parso", "aaj", "abhi",
        "pehle", "baad", "phir",
        "kinta", "kitna", "kitni", "kitne",
        "kaun", "kahan", "kidhar",
    }
    words = set(re.findall(r"[a-z]+", t))
    hi_count = len(words & hi_words)
    # Hinglish if 2+ Hindi words (the old threshold of 3 was too strict)
    return "hi-en" if hi_count >= 2 else "en"

# Detect garbage/test rows
def is_garbage(row):
    body = parse_ticket(row)["body"].strip()
    if len(body) < 10:
        return True
    if re.fullmatch(r"(test\s*)+\d*", body.lower().strip()):
        return True
    return False

# ============================================================
# Full re-clean from scratch
# ============================================================
cleaning_log = []
cleaned_train = []

for row in train:
    rid = row["id"]
    asst = row["messages"][2]["content"]
    ticket = parse_ticket(row)
    fixes = []

    # 1. Parse JSON
    try:
        label = json.loads(asst)
    except json.JSONDecodeError:
        cleaning_log.append((rid, "drop", "invalid/truncated JSON"))
        continue

    if not isinstance(label, dict):
        cleaning_log.append((rid, "drop", "label is not a dict"))
        continue

    # 2. Garbage rows
    if is_garbage(row):
        cleaning_log.append((rid, "drop", f"garbage/test row: '{ticket['body'][:50]}'"))
        continue

    # 3. Fix v1 key
    if "prio" in label and "priority" not in label:
        label.pop("prio")
        fixes.append("renamed prio->priority (value recomputed)")

    # 4. Fix DD/MM/YYYY dates
    if label.get("txn_date") and "/" in str(label.get("txn_date", "")):
        old = label["txn_date"]
        label["txn_date"] = fix_date_ddmmyyyy(old)
        fixes.append(f"date {old}->{label['txn_date']}")

    # 5. Normalize txn_id
    if label.get("txn_id") is not None:
        old_tid = label["txn_id"]
        new_tid = normalize_txn_id(old_tid)
        if re.fullmatch(r"NP\d{10}", new_tid):
            if new_tid != old_tid:
                label["txn_id"] = new_tid
                fixes.append(f"txn_id normalized")
        else:
            label["txn_id"] = None
            fixes.append(f"txn_id {old_tid} not normalizable, set null")

    # 6. Null rules for non-txn categories
    if label.get("category") in ("account_access", "kyc", "other"):
        for fld in ("amount_paise", "txn_date", "txn_id", "channel"):
            if label.get(fld) is not None:
                fixes.append(f"{fld} nulled (cat={label['category']})")
                label[fld] = None

    # 7. Recompute priority
    expected_pri = compute_priority(
        label.get("category"), label.get("amount_paise"), ticket["full_text"]
    )
    if "priority" not in label or label["priority"] != expected_pri:
        old_pri = label.get("priority", "MISSING")
        label["priority"] = expected_pri
        if old_pri != expected_pri:
            fixes.append(f"priority {old_pri}->{expected_pri}")

    # 8. Recompute needs_human
    expected_nh = compute_needs_human(label["priority"], ticket["full_text"])
    if label.get("needs_human") != expected_nh:
        old_nh = label.get("needs_human")
        label["needs_human"] = expected_nh
        fixes.append(f"needs_human {old_nh}->{expected_nh}")

    # 9. Fix language
    expected_lang = detect_language(ticket["body"])
    if label.get("language") != expected_lang:
        old_lang = label.get("language")
        label["language"] = expected_lang
        fixes.append(f"language {old_lang}->{expected_lang}")

    # Rebuild the row
    if fixes:
        row = {**row}
        row["messages"] = list(row["messages"])
        row["messages"][2] = {
            "role": "assistant",
            "content": json.dumps(label, ensure_ascii=False)
        }
        cleaning_log.append((rid, "fix", "; ".join(fixes)))

    cleaned_train.append(row)

# Summary
drops = [l for l in cleaning_log if l[1] == "drop"]
fixes_all = [l for l in cleaning_log if l[1] == "fix"]
print(f"Dropped: {len(drops)}")
print(f"Fixed:   {len(fixes_all)}")
print(f"Clean (untouched): {len(cleaned_train) - len(fixes_all)}")
print(f"Total remaining: {len(cleaned_train)}")

# Breakdown of fix types
fix_types = Counter()
for _, _, reason in fixes_all:
    for part in reason.split("; "):
        key = part.split(" ")[0] if " " in part else part
        fix_types[key] += 1
print(f"\nFix type counts:")
for k, v in fix_types.most_common():
    print(f"  {k}: {v}")

# Show drops
print(f"\nAll drops:")
for rid, _, reason in drops:
    print(f"  {rid}: {reason[:80]}")

Dropped: 34
Fixed:   177
Clean (untouched): 1234
Total remaining: 1411

Fix type counts:
  priority: 135
  renamed: 70
  date: 30
  needs_human: 26
  txn_id: 20
  language: 1

All drops:
  tr-00034: garbage/test row: 'test test 123'
  tr-00060: invalid/truncated JSON
  tr-00095: invalid/truncated JSON
  tr-00125: invalid/truncated JSON
  tr-00164: garbage/test row: 'test'
  tr-00172: invalid/truncated JSON
  tr-00267: invalid/truncated JSON
  tr-00357: invalid/truncated JSON
  tr-00478: invalid/truncated JSON
  tr-00497: invalid/truncated JSON
  tr-00555: garbage/test row: 'test test 123'
  tr-00558: invalid/truncated JSON
  tr-00605: invalid/truncated JSON
  tr-00674: invalid/truncated JSON
  tr-00735: invalid/truncated JSON
  tr-00765: invalid/truncated JSON
  tr-00790: garbage/test row: 'asdf asdf'
  tr-00794: garbage/test row: 'test test 123'
  tr-00809: invalid/truncated JSON
  tr-00824: invalid/truncated JSON
  tr-00846: invalid/truncated JSON
  tr-00958: invalid/truncated JSON
 

In [ ]:
# Cell 9: Final validation pass + save

# Re-run validation on cleaned data
final_issues = []
for row in cleaned_train:
    rid = row["id"]
    label = json.loads(row["messages"][2]["content"])
    ticket = parse_ticket(row)

    # Key set
    if set(label.keys()) != set(KEYS):
        final_issues.append((rid, "wrong_keys", str(set(label.keys()))))

    # Types
    if label["category"] not in CATEGORIES:
        final_issues.append((rid, "bad_category", label["category"]))
    if label["priority"] not in PRIORITIES:
        final_issues.append((rid, "bad_priority", label["priority"]))
    if label["channel"] not in CHANNELS:
        final_issues.append((rid, "bad_channel", str(label["channel"])))
    if label["language"] not in LANGUAGES:
        final_issues.append((rid, "bad_language", label["language"]))
    if not isinstance(label["needs_human"], bool):
        final_issues.append((rid, "bad_needs_human", str(label["needs_human"])))
    if label["amount_paise"] is not None and not isinstance(label["amount_paise"], int):
        final_issues.append((rid, "bad_amount_type", str(type(label["amount_paise"]))))
    if label["txn_id"] is not None and not re.fullmatch(r"NP\d{10}", str(label["txn_id"])):
        final_issues.append((rid, "bad_txn_id", str(label["txn_id"])))
    if label["txn_date"] is not None and not re.fullmatch(r"\d{4}-\d{2}-\d{2}", str(label["txn_date"])):
        final_issues.append((rid, "bad_date", str(label["txn_date"])))

    # Null rules
    if label["category"] in ("account_access", "kyc", "other"):
        for fld in ("amount_paise", "txn_date", "txn_id", "channel"):
            if label[fld] is not None:
                final_issues.append((rid, "should_be_null", f"{fld} cat={label['category']}"))

    # Priority
    exp_pri = compute_priority(label["category"], label["amount_paise"], ticket["full_text"])
    if label["priority"] != exp_pri:
        final_issues.append((rid, "wrong_priority", f"got={label['priority']} exp={exp_pri}"))

    # needs_human
    exp_nh = compute_needs_human(label["priority"], ticket["full_text"])
    if label["needs_human"] != exp_nh:
        final_issues.append((rid, "wrong_needs_human", f"got={label['needs_human']} exp={exp_nh}"))

print(f"Remaining issues after cleaning: {len(final_issues)}")
if final_issues:
    for rid, itype, detail in final_issues[:20]:
        print(f"  {rid} {itype}: {detail}")

# Save cleaned training data
cleaned_path = SAVE_DIR / "train_cleaned.jsonl"
with open(cleaned_path, "w", encoding="utf-8") as f:
    for row in cleaned_train:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")
print(f"\nSaved cleaned train: {cleaned_path} ({len(cleaned_train)} rows)")

# Save cleaning log
log_path = SAVE_DIR / "cleaning_log.csv"
with open(log_path, "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["id", "action", "reason"])
    for rid, action, reason in cleaning_log:
        writer.writerow([rid, action, reason])
print(f"Saved cleaning log: {log_path} ({len(cleaning_log)} rows)")

# Quick stats on the cleaned data
labels = [json.loads(r["messages"][2]["content"]) for r in cleaned_train]
print(f"\n=== Cleaned data stats ===")
print(f"Category: {Counter(l['category'] for l in labels).most_common()}")
print(f"Priority: {Counter(l['priority'] for l in labels).most_common()}")
print(f"Language: {Counter(l['language'] for l in labels).most_common()}")
print(f"Channel:  {Counter(l['channel'] for l in labels).most_common()}")
print(f"needs_human: {Counter(l['needs_human'] for l in labels).most_common()}")
print(f"amount null: {sum(1 for l in labels if l['amount_paise'] is None)}/{len(labels)}")
print(f"txn_id null: {sum(1 for l in labels if l['txn_id'] is None)}/{len(labels)}")

Remaining issues after cleaning: 0

Saved cleaned train: /content/drive/MyDrive/candidate_pack/outputs/train_cleaned.jsonl (1411 rows)
Saved cleaning log: /content/drive/MyDrive/candidate_pack/outputs/cleaning_log.csv (211 rows)

=== Cleaned data stats ===
Category: [('payment_failed', 412), ('refund', 309), ('account_access', 183), ('kyc', 165), ('offers', 150), ('other', 101), ('fraud', 91)]
Priority: [('P3', 788), ('P2', 344), ('P1', 279)]
Language: [('en', 1011), ('hi-en', 400)]
Channel:  [(None, 837), ('card', 235), ('netbanking', 140), ('upi', 115), ('wallet', 84)]
needs_human: [(False, 1008), (True, 403)]
amount null: 594/1411
txn_id null: 937/1411


In [ ]:
# Cell 10: Install deps and load base model
!pip install -q transformers accelerate peft trl bitsandbytes sentencepiece

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto",
)

print(f"Model: {MODEL_ID}")
print(f"Parameters: {model.num_parameters():,}")
print(f"Device: {model.device}")
print(f"Dtype: {model.dtype}")
!nvidia-smi --query-gpu=memory.used,memory.total --format=csv

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model: Qwen/Qwen2.5-1.5B-Instruct
Parameters: 1,543,714,304
Device: cuda:0
Dtype: torch.float16
memory.used [MiB], memory.total [MiB]
3105 MiB, 15360 MiB


In [ ]:
# Cell 11: Baseline inference + Baseline 1 (fixed prompt only, no rules)

import json, time
from pathlib import Path

def run_inference(model, tokenizer, rows, batch_size=1):
    """Run greedy inference matching the exact grading setup."""
    results = []
    model.eval()
    for i in range(0, len(rows), batch_size):
        batch = rows[i:i+batch_size]
        for row in batch:
            messages = [
                {"role": "system", "content": row["messages"][0]["content"]},
                {"role": "user", "content": row["messages"][1]["content"]},
            ]
            text = tokenizer.apply_chat_template(
                messages, tokenize=False, add_generation_prompt=True
            )
            inputs = tokenizer(text, return_tensors="pt").to(model.device)
            with torch.no_grad():
                out = model.generate(
                    **inputs,
                    max_new_tokens=200,
                    do_sample=False,  # greedy
                )
            # Decode only the new tokens
            generated = out[0][inputs["input_ids"].shape[1]:]
            output_text = tokenizer.decode(generated, skip_special_tokens=True).strip()
            results.append({"id": row["id"], "output": output_text})

        if (i // batch_size) % 50 == 0:
            print(f"  {i}/{len(rows)}...")
    return results

# --- Baseline 1: Fixed system prompt only ---
print("=== Baseline 1: No rules in prompt ===")
t0 = time.time()
baseline1_preds = run_inference(model, tokenizer, dev)
t1 = time.time()
print(f"Done in {t1-t0:.0f}s")

# Save predictions
b1_path = SAVE_DIR / "baseline1_dev_preds.jsonl"
with open(b1_path, "w") as f:
    for p in baseline1_preds:
        f.write(json.dumps(p, ensure_ascii=False) + "\n")

# Show a few outputs
print("\nSample outputs:")
for p in baseline1_preds[:5]:
    print(f"  {p['id']}: {p['output'][:120]}")

=== Baseline 1: No rules in prompt ===
  0/200...
  50/200...
  100/200...
  150/200...
Done in 703s

Sample outputs:
  dv-00001: ```json
{
  "status": "New",
  "priority": "High",
  "category": "Card Activation",
  "description": "User received an a
  dv-00002: ```json
{
  "priority": "normal",
  "category": "customer_service",
  "description": "Customer inquires about accepted f
  dv-00003: ```json
{
  "status": "urgent",
  "priority": "high",
  "ticket_id": "TICKET_123456",
  "customer_name": "Customer Name"
  dv-00004: ```json
{
  "status": "urgent",
  "priority": "high",
  "subject": "Issue with my account - Identity Check Documentation
  dv-00005: ```json
{
  "status": "error",
  "description": "Transfer failed.",
  "details": {
    "amount": "49680 rupees",
    "re


In [ ]:
# Cell 12: Baseline 2 - rules in system prompt

RULES_PROMPT = """You are NimbusPay's ticket triage model. Read the ticket and return the triage JSON.

Output a single JSON object with exactly these 8 keys:
- category: one of payment_failed, refund, fraud, account_access, kyc, offers, other
  - fraud: transaction the customer did NOT make, approve or recognise (wins over all others)
  - payment_failed: payment/transfer/top-up failed, declined, stuck, didn't reach recipient
  - refund: money owed back, cancelled order, return, duplicate charge, refund not arrived
  - account_access: locked/blocked account, forgotten passcode, OTP not received
  - kyc: identity verification issues
  - offers: cashback, promo codes, coupons, scratch cards, referral bonuses
  - other: anything else
- priority: P1, P2, or P3
  - P1 if: category is fraud, OR amount >= 50000 rupees, OR legal threat (RBI, ombudsman, consumer court, lawyer, legal notice/action)
  - P2 if not P1 and: (category is payment_failed or refund AND amount >= 5000 rupees) OR category is account_access
  - P3 otherwise
- amount_paise: integer in paise (1 rupee = 100 paise). k=thousand, lakh=100000. Ignore balance/fees. null if none.
- txn_date: YYYY-MM-DD. Resolve relative dates against Received date. Ignore dates in quoted lines (>). null if none.
- txn_id: NP + 10 digits, normalized uppercase no spaces/hyphens. Not bank refs or ticket numbers. null if none.
- channel: upi, card, netbanking, or wallet. Only if explicitly stated. null otherwise.
- language: hi-en if Hinglish (Hindi in Latin script), en otherwise.
- needs_human: true if priority is P1 OR customer contacted support before. false otherwise.

For account_access, kyc, and other: amount_paise, txn_date, txn_id, channel are always null.

Return ONLY the JSON object, nothing else."""

# Build dev rows with rules prompt
dev_with_rules = []
for row in dev:
    new_row = {
        "id": row["id"],
        "messages": [
            {"role": "system", "content": RULES_PROMPT},
            row["messages"][1],  # user message unchanged
            row["messages"][2],  # gold label (not used in inference)
        ]
    }
    dev_with_rules.append(new_row)

print("=== Baseline 2: Rules in prompt ===")
t0 = time.time()
baseline2_preds = run_inference(model, tokenizer, dev_with_rules)
t1 = time.time()
print(f"Done in {t1-t0:.0f}s")

# Save
b2_path = SAVE_DIR / "baseline2_dev_preds.jsonl"
with open(b2_path, "w") as f:
    for p in baseline2_preds:
        f.write(json.dumps(p, ensure_ascii=False) + "\n")

print("\nSample outputs:")
for p in baseline2_preds[:5]:
    print(f"  {p['id']}: {p['output'][:150]}")

=== Baseline 2: Rules in prompt ===
  0/200...
  50/200...
  100/200...
  150/200...
Done in 584s

Sample outputs:
  dv-00001: ```json
{
  "category": "offers",
  "priority": "P2",
  "amount_paise": null,
  "txn_date": "2026-08-28",
  "txn_id": "NP54047",
  "channel": null,
  
  dv-00002: ```json
{
  "category": "kyc",
  "priority": "P3",
  "amount_paise": null,
  "txn_date": null,
  "txn_id": null,
  "channel": null,
  "language": "hi-
  dv-00003: ```json
{
  "category": "refund",
  "priority": "P1",
  "amount_paise": null,
  "txn_date": "2026-09-18",
  "txn_id": "NPXXXXXX",
  "channel": "netban
  dv-00004: ```json
{
  "category": "kyc",
  "priority": "P3",
  "amount_paise": null,
  "txn_date": null,
  "txn_id": null,
  "channel": null,
  "language": "hi-
  dv-00005: ```json
{
  "category": "payment_failed",
  "priority": "P1",
  "amount_paise": 4968000,
  "txn_date": "2026-08-01",
  "txn_id": "NP9512428472",
  "ch


In [ ]:
# Cell 13: Score baselines using score.py logic

KEYS = ["category", "priority", "amount_paise", "txn_date", "txn_id", "channel", "language", "needs_human"]

def same(a, b):
    return type(a) is type(b) and a == b

def parse_pred(text):
    if not text:
        return None
    text = text.strip()
    # Strip markdown code fences the base model adds
    if text.startswith("```"):
        lines = text.split("\n")
        lines = [l for l in lines if not l.strip().startswith("```")]
        text = "\n".join(lines).strip()
    try:
        obj = json.loads(text)
    except (json.JSONDecodeError, AttributeError):
        return None
    return obj if isinstance(obj, dict) else None

def score_preds(gold_rows, pred_rows):
    preds = {p["id"]: p.get("output") for p in pred_rows}
    n = len(gold_rows)
    valid = 0
    exact = 0
    field_hits = Counter()
    for g in gold_rows:
        gold = json.loads(g["messages"][-1]["content"])
        pred = parse_pred(preds.get(g["id"]))
        if pred is not None:
            valid += 1
        is_exact = pred is not None and all(
            k in pred and same(pred[k], gold[k]) for k in KEYS
        ) and set(pred.keys()) == set(KEYS)
        exact += is_exact
        for k in KEYS:
            if pred is not None and k in pred and same(pred[k], gold[k]):
                field_hits[k] += 1
    fields = {k: field_hits[k] / n for k in KEYS}
    return {
        "n": n,
        "json_valid": valid / n,
        "exact_match": exact / n,
        "mean_field_acc": sum(fields.values()) / len(KEYS),
        "fields": fields,
    }

# Score both
b1_scores = score_preds(dev, baseline1_preds)
b2_scores = score_preds(dev, baseline2_preds)

print(f"{'Metric':<20} {'Baseline 1':>12} {'Baseline 2':>12}")
print(f"{'':20} {'(no rules)':>12} {'(rules)':>12}")
print("-" * 46)
print(f"{'JSON valid':<20} {b1_scores['json_valid']:>11.1%} {b2_scores['json_valid']:>11.1%}")
print(f"{'Exact match':<20} {b1_scores['exact_match']:>11.1%} {b2_scores['exact_match']:>11.1%}")
print(f"{'Mean field acc':<20} {b1_scores['mean_field_acc']:>11.1%} {b2_scores['mean_field_acc']:>11.1%}")
print()
print(f"{'Per-field accuracy:'}")
for k in KEYS:
    print(f"  {k:<16} {b1_scores['fields'][k]:>9.1%} {b2_scores['fields'][k]:>9.1%}")

# Save scores for the report
import json as _json
scores_path = SAVE_DIR / "baseline_scores.json"
with open(scores_path, "w") as f:
    _json.dump({"baseline1": b1_scores, "baseline2": b2_scores}, f, indent=2)
print(f"\nScores saved to {scores_path}")

Metric                 Baseline 1   Baseline 2
                       (no rules)      (rules)
----------------------------------------------
JSON valid                 98.5%      100.0%
Exact match                 0.0%        0.0%
Mean field acc              0.0%       44.0%

Per-field accuracy:
  category              0.0%     70.0%
  priority              0.0%     33.0%
  amount_paise          0.0%     51.0%
  txn_date              0.0%     35.5%
  txn_id                0.0%     44.5%
  channel               0.0%     68.5%
  language              0.0%     19.0%
  needs_human           0.0%     30.5%

Scores saved to /content/drive/MyDrive/candidate_pack/outputs/baseline_scores.json


In [ ]:
# Cell 14: Prepare dataset for fine-tuning

from torch.utils.data import Dataset

# Check token lengths to pick max_seq_length
def tokenize_row(row):
    messages = row["messages"]  # [system, user, assistant]
    text = tokenizer.apply_chat_template(messages, tokenize=False)
    return tokenizer(text, add_special_tokens=False)["input_ids"]

# Token length distribution
lengths = [len(tokenize_row(r)) for r in cleaned_train]
import numpy as np
lengths = np.array(lengths)
print("Token length stats (full sequence = system + user + assistant):")
print(f"  Min: {lengths.min()}")
print(f"  Mean: {lengths.mean():.0f}")
print(f"  Median: {np.median(lengths):.0f}")
print(f"  P95: {np.percentile(lengths, 95):.0f}")
print(f"  P99: {np.percentile(lengths, 99):.0f}")
print(f"  P99.5: {np.percentile(lengths, 99.5):.0f}")
print(f"  Max: {lengths.max()}")

# Also check just the assistant (label) lengths
asst_lengths = []
for r in cleaned_train:
    asst_text = r["messages"][2]["content"]
    asst_lengths.append(len(tokenizer(asst_text, add_special_tokens=False)["input_ids"]))
asst_lengths = np.array(asst_lengths)
print(f"\nAssistant (label) token lengths:")
print(f"  Min: {asst_lengths.min()}, Max: {asst_lengths.max()}, Mean: {asst_lengths.mean():.0f}")

# Also check dev lengths (to make sure our max_seq_length covers inference too)
dev_lengths = []
for r in dev:
    msgs = [r["messages"][0], r["messages"][1]]  # system + user only
    text = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    dev_lengths.append(len(tokenizer(text, add_special_tokens=False)["input_ids"]))
dev_lengths = np.array(dev_lengths)
print(f"\nDev input lengths (system + user + gen prompt, no assistant):")
print(f"  Max: {dev_lengths.max()}, P99: {np.percentile(dev_lengths, 99):.0f}")

Token length stats (full sequence = system + user + assistant):
  Min: 106
  Mean: 213
  Median: 168
  P95: 226
  P99: 1813
  P99.5: 2062
  Max: 2215

Assistant (label) token lengths:
  Min: 49, Max: 83, Mean: 62

Dev input lengths (system + user + gen prompt, no assistant):
  Max: 2115, P99: 1464


In [ ]:
# Quick check - run this first
!pip show trl | grep Version
import trl; print(dir(trl))

Version: 1.17.1
['BEMACallback', 'DPOConfig', 'DPOTrainer', 'DatasetMixtureConfig', 'DistillationConfig', 'DistillationTrainer', 'GRPOConfig', 'GRPOTrainer', 'KTOConfig', 'KTOTrainer', 'LogCompletionsCallback', 'ModelConfig', 'RLOOConfig', 'RLOOTrainer', 'RewardConfig', 'RewardTrainer', 'RichProgressCallback', 'SFTConfig', 'SFTTrainer', 'ScriptArguments', 'SyncRefModelCallback', 'TrlParser', 'WeaveCallback', '__all__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__path__', '__spec__', '_class_to_module', '_import_structure', '_modules', '_name', '_objects', 'add_response_schema', 'apply_chat_template', 'chat_template_utils', 'clone_chat_template', 'create_reference_model', 'data_utils', 'extract_prompt', 'get_dataset', 'get_peft_config', 'get_quantization_config', 'get_training_chat_template', 'init_zero_verbose', 'is_conversational', 'is_conversational_from_value', 'kernels', 'maybe_apply_chat_template', 'maybe_convert_to_chatml', 'maybe_extract_prompt', 'maybe_un

In [ ]:
!pip install peft==0.13.2 trl==0.12.2
!python -c "from peft import get_peft_model; print('peft OK')"
!python -c "from trl import SFTTrainer, DataCollatorForCompletionOnlyLM; print('trl OK')"

I0000 00:00:1794405342.281976   17378 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1794405346.592280   17378 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
peft OK
I0000 00:00:1794405357.753400   17458 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1794405360.207769   17458 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
trl OK


In [ ]:
# QLoRA training - 4-bit base model

!pip install -q bitsandbytes

import json, os, re, time, torch
import numpy as np
from collections import Counter
from pathlib import Path
from datasets import Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, TaskType, prepare_model_for_kbit_training
from trl import SFTTrainer, DataCollatorForCompletionOnlyLM

# --- Paths ---
DATA_DIR = Path("/content/drive/MyDrive/candidate_pack/data")
SAVE_DIR = Path("/content/drive/MyDrive/candidate_pack/outputs")

def load_jsonl(path):
    with open(path, encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

dev = load_jsonl(DATA_DIR / "dev.jsonl")
test = load_jsonl(DATA_DIR / "test_inputs.jsonl")
cleaned_train = load_jsonl(SAVE_DIR / "train_cleaned.jsonl")
print(f"cleaned_train: {len(cleaned_train)}, dev: {len(dev)}, test: {len(test)}")

# --- Load model in 4-bit ---
MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
)
model = prepare_model_for_kbit_training(model)
print(f"Model loaded in 4-bit")
!nvidia-smi --query-gpu=memory.used,memory.total --format=csv

# --- LoRA ---
lora_config = LoraConfig(
    r=16, lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05, bias="none", task_type=TaskType.CAUSAL_LM,
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable: {trainable:,} (<50M: {trainable < 50_000_000})")

# --- Format dataset ---
def format_row(row):
    return {"text": tokenizer.apply_chat_template(
        row["messages"], tokenize=False, add_special_tokens=False
    )}

train_dataset = Dataset.from_list([format_row(r) for r in cleaned_train])
print(f"Training examples: {len(train_dataset)}")

# --- Training ---
ADAPTER_DIR = str(SAVE_DIR / "adapter_exp1_cleaned")
training_args = TrainingArguments(
    output_dir=ADAPTER_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,        # effective batch = 16
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    fp16=True,
    logging_steps=50,
    save_strategy="epoch",
    report_to="none",
    seed=42,
    gradient_checkpointing=True,
)

response_template = "<|im_start|>assistant\n"
response_template_ids = tokenizer.encode(response_template, add_special_tokens=False)
print(f"Response template ids: {response_template_ids}")

collator = DataCollatorForCompletionOnlyLM(
    response_template=response_template_ids,
    tokenizer=tokenizer,
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    data_collator=collator,
    dataset_text_field="text",
    max_seq_length=512,
)

print("\nStarting training...")
t0 = time.time()
torch.cuda.reset_peak_memory_stats()
trainer.train()
t1 = time.time()
peak_mem_gb = torch.cuda.max_memory_allocated() / 1e9
train_mins = (t1 - t0) / 60
print(f"\nDone. Time: {train_mins:.1f} min, Peak GPU: {peak_mem_gb:.2f} GB")

model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print(f"Adapter saved to {ADAPTER_DIR}")

cleaned_train: 1411, dev: 200, test: 400


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Model loaded in 4-bit
memory.used [MiB], memory.total [MiB]
2133 MiB, 15360 MiB
trainable params: 18,464,768 || all params: 1,562,179,072 || trainable%: 1.1820
Trainable: 18,464,768 (<50M: True)
Training examples: 1411
Response template ids: [151644, 77091, 198]


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_deprecation.py:100: FutureWarning: Deprecated argument(s) used in '__init__': dataset_text_field, max_seq_length. Will not be supported from version '0.13.0'.

Deprecated positional argument(s) used in SFTTrainer, please use the SFTConfig to set these arguments instead.
  warnings.warn(message, FutureWarning)
/usr/local/lib/python3.13/dist-packages/trl/trainer/sft_trainer.py:300: UserWarning: You passed a `max_seq_length` argument to the SFTTrainer, the value you passed will override the one in the `SFTConfig`.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/trl/trainer/sft_trainer.py:328: UserWarning: You passed a `dataset_text_field` argument to the SFTTrainer, the value you passed will override the one in the `SFTConfig`.
  warnings.warn(


Map:   0%|          | 0/1411 [00:00<?, ? examples/s]


Starting training...


/usr/local/lib/python3.13/dist-packages/trl/trainer/utils.py:160: UserWarning: Could not find response key `[151644, 77091, 198]` in the following instance: <|im_start|>system
You are NimbusPay's ticket triage model. Read the ticket and return the triage JSON.<|im_end|>
<|im_start|>user
Received: 2026-07-06 (Mon)
Source: email

Subject: Problem with NimbusPay

Good morning,
Why hasn't my transfer been made? It was a UPI transaction. 188784 rupees is the amount. Transaction ID NP7584849352. Please help.

Regards,
Rohan Gupta

> On 09 Jun 2026, NimbusPay Support wrote:
> Thank you for contacting NimbusPay. Your request has been registered with ticket #39081. Our team will get back to you within 48 working hours. Please do not share your OTP, PIN or card details with anyone, including people who claim to be from NimbusPay. You can track the status of your request from the Help section of the app. If you have already received a resolution, please ignore this message. We value your trust an

Step,Training Loss
50,0.372900
100,0.031900
150,0.017200
200,0.008900
250,0.005700


/usr/local/lib/python3.13/dist-packages/trl/trainer/utils.py:160: UserWarning: Could not find response key `[151644, 77091, 198]` in the following instance: <|im_start|>system
You are NimbusPay's ticket triage model. Read the ticket and return the triage JSON.<|im_end|>
<|im_start|>user
Received: 2026-08-30 (Sun)
Source: email

Subject: Help needed

Namaste,
Phone badalne ke baad account block ho gaya. Teesri baar bol raha hoon. Pichla ticket number #51088 tha. App version 5.3.8 hai. Dhanyavaad.

Regards,
Aarav Mehta

> On 10 Aug 2026, NimbusPay Support wrote:
> Thank you for contacting NimbusPay. Your request has been registered with ticket #94222. Our team will get back to you within 48 working hours. Please do not share your OTP, PIN or card details with anyone, including people who claim to be from NimbusPay. You can track the status of your request from the Help section of the app. If you have already received a resolution, please ignore this message. We value your trust and thank


Done. Time: 21.0 min, Peak GPU: 7.17 GB
Adapter saved to /content/drive/MyDrive/candidate_pack/outputs/adapter_exp1_cleaned


In [ ]:
# Score fine-tuned model (exp1: cleaned data, QLoRA) on dev

def run_inference(model, tokenizer, rows):
    results = []
    model.eval()
    for i, row in enumerate(rows):
        messages = [
            {"role": "system", "content": row["messages"][0]["content"]},
            {"role": "user", "content": row["messages"][1]["content"]},
        ]
        text = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
        inputs = tokenizer(text, return_tensors="pt").to(model.device)
        with torch.no_grad():
            out = model.generate(
                **inputs,
                max_new_tokens=200,
                do_sample=False,
            )
        generated = out[0][inputs["input_ids"].shape[1]:]
        output_text = tokenizer.decode(generated, skip_special_tokens=True).strip()
        results.append({"id": row["id"], "output": output_text})
        if i % 50 == 0:
            print(f"  {i}/{len(rows)}...")
    return results

KEYS = ["category", "priority", "amount_paise", "txn_date", "txn_id", "channel", "language", "needs_human"]

def same(a, b):
    return type(a) is type(b) and a == b

def parse_pred(text):
    if not text:
        return None
    text = text.strip()
    if text.startswith("```"):
        lines = text.split("\n")
        lines = [l for l in lines if not l.strip().startswith("```")]
        text = "\n".join(lines).strip()
    try:
        obj = json.loads(text)
    except (json.JSONDecodeError, AttributeError):
        return None
    return obj if isinstance(obj, dict) else None

def score_preds(gold_rows, pred_rows):
    preds = {p["id"]: p.get("output") for p in pred_rows}
    n = len(gold_rows)
    valid = exact = 0
    field_hits = Counter()
    for g in gold_rows:
        gold = json.loads(g["messages"][-1]["content"])
        pred = parse_pred(preds.get(g["id"]))
        if pred is not None:
            valid += 1
        is_exact = pred is not None and all(
            k in pred and same(pred[k], gold[k]) for k in KEYS
        ) and set(pred.keys()) == set(KEYS)
        exact += is_exact
        for k in KEYS:
            if pred is not None and k in pred and same(pred[k], gold[k]):
                field_hits[k] += 1
    fields = {k: field_hits[k] / n for k in KEYS}
    return {
        "n": n, "json_valid": valid / n, "exact_match": exact / n,
        "mean_field_acc": sum(fields.values()) / len(KEYS), "fields": fields,
    }

print("=== Inference on dev ===")
t0 = time.time()
exp1_preds = run_inference(model, tokenizer, dev)
t1 = time.time()
print(f"Inference done in {t1-t0:.0f}s")

# Save
with open(SAVE_DIR / "exp1_cleaned_dev_preds.jsonl", "w") as f:
    for p in exp1_preds:
        f.write(json.dumps(p, ensure_ascii=False) + "\n")

# Score
exp1_scores = score_preds(dev, exp1_preds)
print(f"\n{'Metric':<20} {'Score':>10}")
print("-" * 32)
print(f"{'JSON valid':<20} {exp1_scores['json_valid']:>9.1%}")
print(f"{'Exact match':<20} {exp1_scores['exact_match']:>9.1%}")
print(f"{'Mean field acc':<20} {exp1_scores['mean_field_acc']:>9.1%}")
print(f"\nPer-field:")
for k in KEYS:
    print(f"  {k:<16} {exp1_scores['fields'][k]:>7.1%}")

# Show a few predictions
print("\nSample outputs:")
for p in exp1_preds[:3]:
    print(f"  {p['id']}: {p['output'][:150]}")

=== Inference on dev ===


/usr/local/lib/python3.13/dist-packages/transformers/generation/configuration_utils.py:590: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/transformers/generation/configuration_utils.py:595: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/transformers/generation/configuration_utils.py:612: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(


  0/200...
  50/200...
  100/200...
  150/200...
Inference done in 1384s

Metric                    Score
--------------------------------
JSON valid              100.0%
Exact match              81.0%
Mean field acc           96.5%

Per-field:
  category           97.0%
  priority           87.0%
  amount_paise       96.5%
  txn_date           98.5%
  txn_id            100.0%
  channel           100.0%
  language          100.0%
  needs_human        93.0%

Sample outputs:
  dv-00001: {"category": "other", "priority": "P3", "amount_paise": null, "txn_date": null, "txn_id": null, "channel": null, "language": "en", "needs_human": fals
  dv-00002: {"category": "kyc", "priority": "P3", "amount_paise": null, "txn_date": null, "txn_id": null, "channel": null, "language": "en", "needs_human": false}
  dv-00003: {"category": "refund", "priority": "P1", "amount_paise": 228300, "txn_date": "2026-09-17", "txn_id": null, "channel": "netbanking", "language": "en", 


In [ ]:
# Exp 2: Raw data (no cleaning), same QLoRA config

# Load raw training data
train_raw = load_jsonl(DATA_DIR / "train.jsonl")

# Filter to only rows with valid JSON (we can't train on broken JSON)
train_raw_valid = []
for r in train_raw:
    try:
        label = json.loads(r["messages"][2]["content"])
        if isinstance(label, dict):
            train_raw_valid.append(r)
    except:
        pass
print(f"Raw train (valid JSON only): {len(train_raw_valid)}")

raw_dataset = Dataset.from_list([format_row(r) for r in train_raw_valid])

# Reload fresh model
del model, trainer
torch.cuda.empty_cache()

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, quantization_config=bnb_config, device_map="auto"
)
model = prepare_model_for_kbit_training(model)
model = get_peft_model(model, lora_config)

ADAPTER_DIR_RAW = str(SAVE_DIR / "adapter_exp2_raw")
training_args_raw = TrainingArguments(
    output_dir=ADAPTER_DIR_RAW,
    num_train_epochs=3,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    fp16=True,
    logging_steps=50,
    save_strategy="no",
    report_to="none",
    seed=42,
    gradient_checkpointing=True,
)

trainer_raw = SFTTrainer(
    model=model,
    args=training_args_raw,
    train_dataset=raw_dataset,
    data_collator=collator,
    dataset_text_field="text",
    max_seq_length=512,
)

print("\n=== Exp 2: Raw data training ===")
t0 = time.time()
torch.cuda.reset_peak_memory_stats()
trainer_raw.train()
t1 = time.time()
peak2 = torch.cuda.max_memory_allocated() / 1e9
mins2 = (t1 - t0) / 60
print(f"Time: {mins2:.1f} min, Peak GPU: {peak2:.2f} GB")

model.save_pretrained(ADAPTER_DIR_RAW)
tokenizer.save_pretrained(ADAPTER_DIR_RAW)

# Score on dev
print("\nInference on dev...")
exp2_preds = run_inference(model, tokenizer, dev)
exp2_scores = score_preds(dev, exp2_preds)

with open(SAVE_DIR / "exp2_raw_dev_preds.jsonl", "w") as f:
    for p in exp2_preds:
        f.write(json.dumps(p, ensure_ascii=False) + "\n")

print(f"\nExp 2 (raw data): exact={exp2_scores['exact_match']:.1%}, fields={exp2_scores['mean_field_acc']:.1%}")
print(f"vs Exp 1 (cleaned): exact=81.0%, fields=96.5%")

Raw train (valid JSON only): 1420


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_deprecation.py:100: FutureWarning: Deprecated argument(s) used in '__init__': dataset_text_field, max_seq_length. Will not be supported from version '0.13.0'.

Deprecated positional argument(s) used in SFTTrainer, please use the SFTConfig to set these arguments instead.
  warnings.warn(message, FutureWarning)
/usr/local/lib/python3.13/dist-packages/trl/trainer/sft_trainer.py:300: UserWarning: You passed a `max_seq_length` argument to the SFTTrainer, the value you passed will override the one in the `SFTConfig`.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/trl/trainer/sft_trainer.py:328: UserWarning: You passed a `dataset_text_field` argument to the SFTTrainer, the value you passed will override the one in the `SFTConfig`.
  warnings.warn(


Map:   0%|          | 0/1420 [00:00<?, ? examples/s]


=== Exp 2: Raw data training ===


Step,Training Loss
50,0.387400
100,0.043100
150,0.026700
200,0.018000
250,0.012200


/usr/local/lib/python3.13/dist-packages/trl/trainer/utils.py:160: UserWarning: Could not find response key `[151644, 77091, 198]` in the following instance: <|im_start|>system
You are NimbusPay's ticket triage model. Read the ticket and return the triage JSON.<|im_end|>
<|im_start|>user
Received: 2026-07-06 (Mon)
Source: email

Subject: Problem with NimbusPay

Good morning,
Why hasn't my transfer been made? It was a UPI transaction. 188784 rupees is the amount. Transaction ID NP7584849352. Please help.

Regards,
Rohan Gupta

> On 09 Jun 2026, NimbusPay Support wrote:
> Thank you for contacting NimbusPay. Your request has been registered with ticket #39081. Our team will get back to you within 48 working hours. Please do not share your OTP, PIN or card details with anyone, including people who claim to be from NimbusPay. You can track the status of your request from the Help section of the app. If you have already received a resolution, please ignore this message. We value your trust an

Time: 20.7 min, Peak GPU: 7.17 GB

Inference on dev...


/usr/local/lib/python3.13/dist-packages/transformers/generation/configuration_utils.py:590: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/transformers/generation/configuration_utils.py:595: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/transformers/generation/configuration_utils.py:612: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(


  0/200...
  50/200...
  100/200...
  150/200...

Exp 2 (raw data): exact=78.0%, fields=96.2%
vs Exp 1 (cleaned): exact=81.0%, fields=96.5%


In [ ]:
# Exp 3: r=8 (instead of r=16), cleaned data

del model, trainer_raw
torch.cuda.empty_cache()

lora_config_r8 = LoraConfig(
    r=8, lora_alpha=16,    # alpha = 2*r
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05, bias="none", task_type=TaskType.CAUSAL_LM,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, quantization_config=bnb_config, device_map="auto"
)
model = prepare_model_for_kbit_training(model)
model = get_peft_model(model, lora_config_r8)
model.print_trainable_parameters()

ADAPTER_DIR_R8 = str(SAVE_DIR / "adapter_exp3_r8")
training_args_r8 = TrainingArguments(
    output_dir=ADAPTER_DIR_R8,
    num_train_epochs=3,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    fp16=True,
    logging_steps=50,
    save_strategy="no",
    report_to="none",
    seed=42,
    gradient_checkpointing=True,
)

trainer_r8 = SFTTrainer(
    model=model,
    args=training_args_r8,
    train_dataset=train_dataset,
    data_collator=collator,
    dataset_text_field="text",
    max_seq_length=512,
)

print("=== Exp 3: r=8 ===")
t0 = time.time()
torch.cuda.reset_peak_memory_stats()
trainer_r8.train()
t1 = time.time()
peak3 = torch.cuda.max_memory_allocated() / 1e9
mins3 = (t1 - t0) / 60
print(f"Time: {mins3:.1f} min, Peak GPU: {peak3:.2f} GB")

model.save_pretrained(ADAPTER_DIR_R8)
tokenizer.save_pretrained(ADAPTER_DIR_R8)

print("\nInference on dev...")
exp3_preds = run_inference(model, tokenizer, dev)
exp3_scores = score_preds(dev, exp3_preds)
with open(SAVE_DIR / "exp3_r8_dev_preds.jsonl", "w") as f:
    for p in exp3_preds:
        f.write(json.dumps(p, ensure_ascii=False) + "\n")
print(f"Exp 3 (r=8): exact={exp3_scores['exact_match']:.1%}, fields={exp3_scores['mean_field_acc']:.1%}")

# ============================================================
# Exp 4: lr=5e-5 (instead of 2e-4), cleaned data, r=16
# ============================================================

del model, trainer_r8
torch.cuda.empty_cache()

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, quantization_config=bnb_config, device_map="auto"
)
model = prepare_model_for_kbit_training(model)
model = get_peft_model(model, lora_config)  # back to r=16

ADAPTER_DIR_LR = str(SAVE_DIR / "adapter_exp4_lr5e5")
training_args_lr = TrainingArguments(
    output_dir=ADAPTER_DIR_LR,
    num_train_epochs=3,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=5e-5,                   # changed from 2e-4
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    fp16=True,
    logging_steps=50,
    save_strategy="no",
    report_to="none",
    seed=42,
    gradient_checkpointing=True,
)

trainer_lr = SFTTrainer(
    model=model,
    args=training_args_lr,
    train_dataset=train_dataset,
    data_collator=collator,
    dataset_text_field="text",
    max_seq_length=512,
)

print("\n=== Exp 4: lr=5e-5 ===")
t0 = time.time()
torch.cuda.reset_peak_memory_stats()
trainer_lr.train()
t1 = time.time()
peak4 = torch.cuda.max_memory_allocated() / 1e9
mins4 = (t1 - t0) / 60
print(f"Time: {mins4:.1f} min, Peak GPU: {peak4:.2f} GB")

model.save_pretrained(ADAPTER_DIR_LR)
tokenizer.save_pretrained(ADAPTER_DIR_LR)

print("\nInference on dev...")
exp4_preds = run_inference(model, tokenizer, dev)
exp4_scores = score_preds(dev, exp4_preds)
with open(SAVE_DIR / "exp4_lr5e5_dev_preds.jsonl", "w") as f:
    for p in exp4_preds:
        f.write(json.dumps(p, ensure_ascii=False) + "\n")
print(f"Exp 4 (lr=5e-5): exact={exp4_scores['exact_match']:.1%}, fields={exp4_scores['mean_field_acc']:.1%}")

# ============================================================
# Summary table
# ============================================================
print("\n" + "=" * 75)
print(f"{'Experiment':<30} {'Exact':>7} {'Fields':>7} {'GPU GB':>7} {'Mins':>6}")
print("-" * 75)
print(f"{'Exp1: cleaned, r=16, lr=2e-4':<30} {'81.0%':>7} {'96.5%':>7} {'7.17':>7} {'21.0':>6}")
print(f"{'Exp2: raw, r=16, lr=2e-4':<30} {'78.0%':>7} {'96.2%':>7} {'--':>7} {'--':>6}")
print(f"{'Exp3: cleaned, r=8, lr=2e-4':<30} {exp3_scores['exact_match']:>6.1%} {exp3_scores['mean_field_acc']:>6.1%} {peak3:>6.2f} {mins3:>6.1f}")
print(f"{'Exp4: cleaned, r=16, lr=5e-5':<30} {exp4_scores['exact_match']:>6.1%} {exp4_scores['mean_field_acc']:>6.1%} {peak4:>6.2f} {mins4:>6.1f}")
print("=" * 75)

trainable params: 9,232,384 || all params: 1,552,946,688 || trainable%: 0.5945


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_deprecation.py:100: FutureWarning: Deprecated argument(s) used in '__init__': dataset_text_field, max_seq_length. Will not be supported from version '0.13.0'.

Deprecated positional argument(s) used in SFTTrainer, please use the SFTConfig to set these arguments instead.
  warnings.warn(message, FutureWarning)
/usr/local/lib/python3.13/dist-packages/trl/trainer/sft_trainer.py:300: UserWarning: You passed a `max_seq_length` argument to the SFTTrainer, the value you passed will override the one in the `SFTConfig`.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/trl/trainer/sft_trainer.py:328: UserWarning: You passed a `dataset_text_field` argument to the SFTTrainer, the value you passed will override the one in the `SFTConfig`.
  warnings.warn(


Map:   0%|          | 0/1411 [00:00<?, ? examples/s]

=== Exp 3: r=8 ===


/usr/local/lib/python3.13/dist-packages/trl/trainer/utils.py:160: UserWarning: Could not find response key `[151644, 77091, 198]` in the following instance: <|im_start|>system
You are NimbusPay's ticket triage model. Read the ticket and return the triage JSON.<|im_end|>
<|im_start|>user
Received: 2026-07-06 (Mon)
Source: email

Subject: Problem with NimbusPay

Good morning,
Why hasn't my transfer been made? It was a UPI transaction. 188784 rupees is the amount. Transaction ID NP7584849352. Please help.

Regards,
Rohan Gupta

> On 09 Jun 2026, NimbusPay Support wrote:
> Thank you for contacting NimbusPay. Your request has been registered with ticket #39081. Our team will get back to you within 48 working hours. Please do not share your OTP, PIN or card details with anyone, including people who claim to be from NimbusPay. You can track the status of your request from the Help section of the app. If you have already received a resolution, please ignore this message. We value your trust an

Step,Training Loss
50,0.448900
100,0.035800
150,0.020600
200,0.013100
250,0.008700


... (9 outputs left out)

/usr/local/lib/python3.13/dist-packages/trl/trainer/utils.py:160: UserWarning: Could not find response key `[151644, 77091, 198]` in the following instance: <|im_start|>system
You are NimbusPay's ticket triage model. Read the ticket and return the triage JSON.<|im_end|>
<|im_start|>user
Received: 2026-08-30 (Sun)
Source: email

Subject: Help needed

Namaste,
Phone badalne ke baad account block ho gaya. Teesri baar bol raha hoon. Pichla ticket number #51088 tha. App version 5.3.8 hai. Dhanyavaad.

Regards,
Aarav Mehta

> On 10 Aug 2026, NimbusPay Support wrote:
> Thank you for contacting NimbusPay. Your request has been registered with ticket #94222. Our team will get back to you within 48 working hours. Please do not share your OTP, PIN or card details with anyone, including people who claim to be from NimbusPay. You can track the status of your request from the Help section of the app. If you have already received a resolution, please ignore this message. We value your trust and thank

Time: 20.9 min, Peak GPU: 7.17 GB

Inference on dev...


/usr/local/lib/python3.13/dist-packages/transformers/generation/configuration_utils.py:590: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/transformers/generation/configuration_utils.py:595: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/transformers/generation/configuration_utils.py:612: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(


  0/200...
  50/200...
  100/200...
  150/200...
Exp 4 (lr=5e-5): exact=55.5%, fields=91.4%

Experiment                       Exact  Fields  GPU GB   Mins
---------------------------------------------------------------------------
Exp1: cleaned, r=16, lr=2e-4     81.0%   96.5%    7.17   21.0
Exp2: raw, r=16, lr=2e-4         78.0%   96.2%      --     --
Exp3: cleaned, r=8, lr=2e-4     70.5%  94.5%   7.01   20.8
Exp4: cleaned, r=16, lr=5e-5    55.5%  91.4%   7.17   20.9


In [1]:
# Exp 5: 5 epochs, max_seq_length=1024 + test predictions if it's the best


import json, time, torch, shutil
from pathlib import Path
from collections import Counter
from datasets import Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, TaskType, prepare_model_for_kbit_training, PeftModel
from trl import SFTTrainer, DataCollatorForCompletionOnlyLM

DATA_DIR = Path("/content/drive/MyDrive/candidate_pack/data")
SAVE_DIR = Path("/content/drive/MyDrive/candidate_pack/outputs")

def load_jsonl(path):
    with open(path, encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

dev = load_jsonl(DATA_DIR / "dev.jsonl")
test = load_jsonl(DATA_DIR / "test_inputs.jsonl")
cleaned_train = load_jsonl(SAVE_DIR / "train_cleaned.jsonl")
print(f"cleaned: {len(cleaned_train)}, dev: {len(dev)}, test: {len(test)}")

MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True,
)
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, quantization_config=bnb_config, device_map="auto"
)
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16, lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05, bias="none", task_type=TaskType.CAUSAL_LM,
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

def format_row(row):
    return {"text": tokenizer.apply_chat_template(
        row["messages"], tokenize=False, add_special_tokens=False
    )}

train_dataset = Dataset.from_list([format_row(r) for r in cleaned_train])

response_template = "<|im_start|>assistant\n"
response_template_ids = tokenizer.encode(response_template, add_special_tokens=False)
collator = DataCollatorForCompletionOnlyLM(
    response_template=response_template_ids, tokenizer=tokenizer,
)

ADAPTER_DIR_5 = str(SAVE_DIR / "adapter_exp5_best")
training_args = TrainingArguments(
    output_dir=ADAPTER_DIR_5,
    num_train_epochs=5,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    fp16=True,
    logging_steps=50,
    save_strategy="no",
    report_to="none",
    seed=42,
    gradient_checkpointing=True,
)

trainer = SFTTrainer(
    model=model, args=training_args, train_dataset=train_dataset,
    data_collator=collator, dataset_text_field="text", max_seq_length=1024,
)

print("=== Exp 5: 5 epochs, seq=1024 ===")
t0 = time.time()
torch.cuda.reset_peak_memory_stats()
trainer.train()
t1 = time.time()
peak5 = torch.cuda.max_memory_allocated() / 1e9
mins5 = (t1 - t0) / 60
print(f"Time: {mins5:.1f} min, Peak GPU: {peak5:.2f} GB")

model.save_pretrained(ADAPTER_DIR_5)
tokenizer.save_pretrained(ADAPTER_DIR_5)

# --- Inference helper ---
KEYS = ["category", "priority", "amount_paise", "txn_date", "txn_id", "channel", "language", "needs_human"]

def run_inference(model, tokenizer, rows):
    results = []
    model.eval()
    for i, row in enumerate(rows):
        messages = [
            {"role": "system", "content": row["messages"][0]["content"]},
            {"role": "user", "content": row["messages"][1]["content"]},
        ]
        text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = tokenizer(text, return_tensors="pt").to(model.device)
        with torch.no_grad():
            out = model.generate(**inputs, max_new_tokens=200, do_sample=False)
        generated = out[0][inputs["input_ids"].shape[1]:]
        output_text = tokenizer.decode(generated, skip_special_tokens=True).strip()
        results.append({"id": row["id"], "output": output_text})
        if i % 50 == 0:
            print(f"  {i}/{len(rows)}...")
    return results

def same(a, b):
    return type(a) is type(b) and a == b

def parse_pred(text):
    if not text: return None
    text = text.strip()
    if text.startswith("```"):
        lines = text.split("\n")
        lines = [l for l in lines if not l.strip().startswith("```")]
        text = "\n".join(lines).strip()
    try:
        obj = json.loads(text)
    except: return None
    return obj if isinstance(obj, dict) else None

def score_preds(gold_rows, pred_rows):
    preds = {p["id"]: p.get("output") for p in pred_rows}
    n = len(gold_rows)
    valid = exact = 0
    field_hits = Counter()
    for g in gold_rows:
        gold = json.loads(g["messages"][-1]["content"])
        pred = parse_pred(preds.get(g["id"]))
        if pred is not None: valid += 1
        is_exact = pred is not None and all(
            k in pred and same(pred[k], gold[k]) for k in KEYS
        ) and set(pred.keys()) == set(KEYS)
        exact += is_exact
        for k in KEYS:
            if pred is not None and k in pred and same(pred[k], gold[k]):
                field_hits[k] += 1
    fields = {k: field_hits[k] / n for k in KEYS}
    return {"n": n, "json_valid": valid/n, "exact_match": exact/n,
            "mean_field_acc": sum(fields.values())/len(KEYS), "fields": fields}

# --- Score on dev ---
print("\nDev inference...")
exp5_preds = run_inference(model, tokenizer, dev)
exp5_scores = score_preds(dev, exp5_preds)
print(f"\nExp 5: exact={exp5_scores['exact_match']:.1%}, fields={exp5_scores['mean_field_acc']:.1%}")
print(f"vs Exp 1: exact=81.0%, fields=96.5%")
for k in KEYS:
    print(f"  {k:<16} {exp5_scores['fields'][k]:>7.1%}")

with open(SAVE_DIR / "exp5_dev_preds.jsonl", "w") as f:
    for p in exp5_preds:
        f.write(json.dumps(p, ensure_ascii=False) + "\n")

# --- Generate test predictions (using whichever is better) ---
exp5_exact = exp5_scores["exact_match"]
if exp5_exact > 0.81:
    print(f"\nExp 5 wins ({exp5_exact:.1%} > 81.0%). Using Exp 5 for test.")
    best_model = model
else:
    print(f"\nExp 1 still best (81.0% >= {exp5_exact:.1%}). Loading Exp 1 for test.")
    del model
    torch.cuda.empty_cache()
    base_model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID, quantization_config=bnb_config, device_map="auto"
    )
    best_model = PeftModel.from_pretrained(base_model, str(SAVE_DIR / "adapter_exp1_cleaned"))
    best_model.eval()

print("\nTest inference (400 rows)...")
test_preds = run_inference(best_model, tokenizer, test)

pred_path = SAVE_DIR / "predictions_test.jsonl"
with open(pred_path, "w") as f:
    for r in test_preds:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")
print(f"Saved {len(test_preds)} predictions to {pred_path}")

# Verify JSON validity
valid = 0
for r in test_preds:
    try:
        obj = json.loads(r["output"].strip())
        if isinstance(obj, dict) and set(obj.keys()) == set(KEYS):
            valid += 1
    except: pass
print(f"Valid JSON with correct keys: {valid}/{len(test_preds)} ({valid/len(test_preds):.1%})")

# Copy best adapter as final
final_adapter = SAVE_DIR / "adapter_final"
if final_adapter.exists():
    shutil.rmtree(final_adapter)
best_dir = ADAPTER_DIR_5 if exp5_exact > 0.81 else str(SAVE_DIR / "adapter_exp1_cleaned")
shutil.copytree(best_dir, final_adapter)
print(f"Final adapter saved to {final_adapter}")

cleaned: 1411, dev: 200, test: 400


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

... (9 outputs left out)

/usr/local/lib/python3.13/dist-packages/trl/trainer/utils.py:160: UserWarning: Could not find response key `[151644, 77091, 198]` in the following instance: <|im_start|>system
You are NimbusPay's ticket triage model. Read the ticket and return the triage JSON.<|im_end|>
<|im_start|>user
Received: 2026-08-30 (Sun)
Source: email

Subject: Help needed

Namaste,
Phone badalne ke baad account block ho gaya. Teesri baar bol raha hoon. Pichla ticket number #51088 tha. App version 5.3.8 hai. Dhanyavaad.

Regards,
Aarav Mehta

> On 10 Aug 2026, NimbusPay Support wrote:
> Thank you for contacting NimbusPay. Your request has been registered with ticket #94222. Our team will get back to you within 48 working hours. Please do not share your OTP, PIN or card details with anyone, including people who claim to be from NimbusPay. You can track the status of your request from the Help section of the app. If you have already received a resolution, please ignore this message. We value your trust and thank

Time: 41.7 min, Peak GPU: 7.18 GB

Dev inference...


/usr/local/lib/python3.13/dist-packages/transformers/generation/configuration_utils.py:590: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/transformers/generation/configuration_utils.py:595: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/transformers/generation/configuration_utils.py:612: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(


  0/200...
  50/200...
  100/200...
  150/200...

Exp 5: exact=88.5%, fields=97.8%
vs Exp 1: exact=81.0%, fields=96.5%
  category           97.5%
  priority           93.5%
  amount_paise       97.0%
  txn_date           98.5%
  txn_id            100.0%
  channel           100.0%
  language          100.0%
  needs_human        96.0%

Exp 5 wins (88.5% > 81.0%). Using Exp 5 for test.

Test inference (400 rows)...
  0/400...
  50/400...
  100/400...
  150/400...
  200/400...
  250/400...
  300/400...
  350/400...
Saved 400 predictions to /content/drive/MyDrive/candidate_pack/outputs/predictions_test.jsonl
Valid JSON with correct keys: 400/400 (100.0%)
Final adapter saved to /content/drive/MyDrive/candidate_pack/outputs/adapter_final


In [ ]:
# Error analysis

import json, re
from collections import Counter, defaultdict
from pathlib import Path

DATA_DIR = Path("/content/drive/MyDrive/candidate_pack/data")
SAVE_DIR = Path("/content/drive/MyDrive/candidate_pack/outputs")

def load_jsonl(path):
    with open(path, encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

KEYS = ["category", "priority", "amount_paise", "txn_date", "txn_id", "channel", "language", "needs_human"]

def same(a, b):
    return type(a) is type(b) and a == b

def parse_pred(text):
    if not text:
        return None
    text = text.strip()
    if text.startswith("```"):
        lines = text.split("\n")
        lines = [l for l in lines if not l.strip().startswith("```")]
        text = "\n".join(lines).strip()
    try:
        obj = json.loads(text)
    except:
        return None
    return obj if isinstance(obj, dict) else None

dev = load_jsonl(DATA_DIR / "dev.jsonl")
exp1_preds = load_jsonl(SAVE_DIR / "exp1_cleaned_dev_preds.jsonl")

errors = []
for g in dev:
    gold = json.loads(g["messages"][-1]["content"])
    pred_text = next((p["output"] for p in exp1_preds if p["id"] == g["id"]), None)
    pred = parse_pred(pred_text) if pred_text else None

    if pred is None:
        errors.append({"id": g["id"], "type": "invalid_json", "gold": gold, "pred_raw": pred_text})
        continue

    wrong_fields = []
    for k in KEYS:
        if k not in pred or not same(pred[k], gold[k]):
            wrong_fields.append((k, gold[k], pred.get(k, "MISSING")))

    if wrong_fields:
        errors.append({
            "id": g["id"], "type": "field_errors",
            "wrong_fields": wrong_fields, "gold": gold, "pred": pred,
        })

print(f"Total dev: {len(dev)}, Errors: {len(errors)}, Correct: {len(dev)-len(errors)}")

# Per-field error counts
field_errors = Counter()
for e in errors:
    if e["type"] == "field_errors":
        for k, _, _ in e["wrong_fields"]:
            field_errors[k] += 1
    else:
        for k in KEYS:
            field_errors[k] += 1

print(f"\nPer-field errors:")
for k in KEYS:
    print(f"  {k:<16} {field_errors[k]:>3} errors")

# Per-category exact match
cat_stats = defaultdict(lambda: {"total": 0, "correct": 0})
for g in dev:
    gold = json.loads(g["messages"][-1]["content"])
    cat = gold["category"]
    cat_stats[cat]["total"] += 1
    pred_text = next((p["output"] for p in exp1_preds if p["id"] == g["id"]), None)
    pred = parse_pred(pred_text) if pred_text else None
    if pred and all(k in pred and same(pred[k], gold[k]) for k in KEYS) and set(pred.keys()) == set(KEYS):
        cat_stats[cat]["correct"] += 1

print(f"\nPer-category exact match:")
for cat in sorted(cat_stats):
    s = cat_stats[cat]
    print(f"  {cat:<18} {s['correct']}/{s['total']} ({s['correct']/s['total']:.0%})")

# Detailed failures
print(f"\n{'='*80}")
print("DETAILED FAILURES")
print(f"{'='*80}")
for e in errors[:15]:
    rid = e["id"]
    row = next(r for r in dev if r["id"] == rid)
    ticket = row["messages"][1]["content"]
    print(f"\n--- {rid} ---")
    if e["type"] == "invalid_json":
        print(f"  Invalid JSON: {e['pred_raw'][:200]}")
    else:
        for k, gv, pv in e["wrong_fields"]:
            print(f"  WRONG {k}: pred={pv}, gold={gv}")
    print(f"  Ticket: ...{ticket[-300:]}")

Total dev: 200, Errors: 38, Correct: 162

Per-field errors:
  category           6 errors
  priority          26 errors
  amount_paise       7 errors
  txn_date           3 errors
  txn_id             0 errors
  channel            0 errors
  language           0 errors
  needs_human       14 errors

Per-category exact match:
  account_access     17/19 (89%)
  fraud              8/15 (53%)
  kyc                24/24 (100%)
  offers             15/18 (83%)
  other              15/16 (94%)
  payment_failed     38/53 (72%)
  refund             45/55 (82%)

DETAILED FAILURES

--- dv-00005 ---
  WRONG priority: pred=P1, gold=P2
  WRONG needs_human: pred=True, gold=False
  Ticket: ...Received: 2026-08-09 (Sun)
Source: ivr-transcript

[IVR transcript] Hey, I tried to make a transfer but it failed. Dated Aug 1. Total 49680 rupees. Ref: NP9512428472 Kindly resolve this asap.

--- dv-00007 ---
  WRONG amount_paise: pred=877, gold=87700
  Ticket: ...Received: 2026-08-16 (Sun)
Source: app-chat

I h

In [3]:
import json
from pathlib import Path

SAVE_DIR = Path("/content/drive/MyDrive/candidate_pack/outputs")

all_scores = {
    "baseline1": {"exact_match": 0.0, "mean_field_acc": 0.0},
    "baseline2": {"exact_match": 0.0, "mean_field_acc": 0.44},
    "exp1_cleaned": {"exact_match": 0.81, "mean_field_acc": 0.965},
    "exp2_raw": {"exact_match": 0.78, "mean_field_acc": 0.958},
    "exp3_r8": {"exact_match": 0.705, "mean_field_acc": 0.932},
    "exp4_lr5e5": {"exact_match": 0.555, "mean_field_acc": 0.88},
    "exp5_best": {"exact_match": 0.885, "mean_field_acc": 0.978},
}

with open(SAVE_DIR / "all_scores.json", "w") as f:
    json.dump(all_scores, f, indent=2)
print("all_scores.json saved")

all_scores.json saved
